In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql://tongli1997@db.pcic.uvic.ca:5433/crmp?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [4]:
path = '/workspaces/crmprtd/update_round2/input_tables/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges_round2.xlsx', header = 1)   # pandas automatically uses openpyxl


df_name_loc = df[
    (df["ISSUE"].str.strip().str.lower() == "name, location") &
    (df["Network ID"] != 11)
]

df_name_loc =  df_name_loc[['Station ID', 'Unique Names','Unique Location (String)', 'Native ID']].reset_index(drop=True)
df_name_loc["Station ID"] = pd.to_numeric(df_name_loc["Station ID"], errors="coerce").astype("Int64")

df_name_loc


,Station ID,Unique Names,Unique Location (String),Native ID
0,2299,HAGENSBORG II - HAGENSBORG 2,"126.5897 W, 52.3872 N, Elev. 50 m -> 126.5897 ...",904
1,1658,CHILLIWACK NURSERY,"121.6633 W, 49.0983 N, Elev. 295 m -> 121.6628...",74
2,1675,MACHMELL (MB) -> MACHMELL,"126.4517 W, 51.5933 N, Elev. 366 m -> 126.4508...",101
3,1683,BOB QUINN LK -> BOB QUINN LAKE,"126.4517 W, 51.5933 N, Elev. 366 m -> 126.4508...",113
4,1710,INGENIKA PT -> INGENIKA POINT,"125.176 W, 56.9777 N, Elev. 1213 m -> 125.1804...",145
...,...,...,...,...
71,13019,-> Gitanyow,"128.05074 W, 55.24679 N, Elev. null m -> 128.0...",51193
72,13020,-> North Burrage,"130.25056 W, 57.30477 N, Elev. null m -> 130.2...",53131
73,2477,Williston @ Lost Cabin Ck -> Williston Lake at...,"123.7467167 W, 56.05063333 N, Elev. 712 m -> 1...",LST
74,2455,Elk R. ab Campbell Lk,"125.8132639W, 49.85804722N, Elev. 270 m",ELK


In [5]:
import re


def _parse_single_location(loc_str):
    """
    Parse:
    '123.764438 W, 48.51616 N, Elev. 512.14 m'
    '123.764438 W, 48.51616 N, Elev. null m'

    Returns: (lat, lon, elev)
    """
    if pd.isna(loc_str):
        return np.nan, np.nan, np.nan

    pattern = (
        r"([\d\.]+)\s*([EW]),\s*"
        r"([\d\.]+)\s*([NS]),\s*"
        r"Elev\.\s*(null|[\d\.]+)\s*m"
    )

    m = re.search(pattern, loc_str, re.IGNORECASE)
    if not m:
        return np.nan, np.nan, np.nan

    lon_val, lon_dir, lat_val, lat_dir, elev_val = m.groups()

    lon = float(lon_val) * (-1 if lon_dir.upper() == "W" else 1)
    lat = float(lat_val) * (-1 if lat_dir.upper() == "S" else 1)

    elev = np.nan if elev_val.lower() == "null" else float(elev_val)

    return lat, lon, elev


def parse_old_new_lat_lon_elev(loc_str):
    """
    Returns:
    (old_lat, old_lon, old_elev, new_lat, new_lon, new_elev)
    """
    if pd.isna(loc_str):
        return (np.nan,) * 6

    if "->" in loc_str:
        old_str, new_str = map(str.strip, loc_str.split("->", 1))
    else:
        old_str = new_str = loc_str.strip()

    old_lat, old_lon, old_elev = _parse_single_location(old_str)
    new_lat, new_lon, new_elev = _parse_single_location(new_str)

    return (
        old_lat, old_lon, old_elev,
        new_lat, new_lon, new_elev
    )

cols = [
    'old_lat', 'old_lon', 'old_elev',
    'new_lat', 'new_lon', 'new_elev'
]

df_name_loc[cols] = df_name_loc['Unique Location (String)'].apply(
    lambda x: pd.Series(parse_old_new_lat_lon_elev(x))
)

# 
df_name_loc = df_name_loc.drop(columns=['Unique Location (String)'])

In [6]:


def split_station_name(name):
    if pd.isna(name):
        return pd.Series([None, None, 0])

    parts = [p.strip() for p in name.split("->")]

    if len(parts) == 2:
        old_name, new_name = parts
        n_names = 2
    else:
        old_name = new_name = parts[0]
        n_names = 1

    return pd.Series([old_name, new_name, n_names])
    
df_new = df_name_loc.copy()

df_new[['old_name', 'new_name', 'n_names']] = (
    df_new['Unique Names'].apply(split_station_name)
)

df_new = df_new.drop(columns= 'Unique Names')

In [7]:
df_new.iloc[60:]

,Station ID,Native ID,old_lat,old_lon,old_elev,new_lat,new_lon,new_elev,old_name,new_name,n_names
60,2485,NTY,51.147528,-122.793917,1969.0,51.147528,-122.785583,1969.0,North Tyaughton Ck,North Tyaughton Creek,2
61,2487,OSL,56.126667,-124.801389,775.0,56.126614,-124.800697,775.0,Osilinka R. nr End Lk,Osilinka River near End Lake,2
62,2494,PRS,55.080000,-122.900000,700.0,55.083469,-122.916100,700.0,Parsnip R. ab Misinchinka R.,Parsnip River above Misinchinka River,2
63,2498,QIN,49.930000,-125.500000,280.0,49.930019,-125.510340,280.0,Quinsam R. @ Argonaut Brg,Quinsam Riber at Argonaut Bridge,2
64,2508,WOL,49.680556,-125.741667,1490.0,49.704139,-125.679250,1490.0,Wolf R. Upper,Wolf River Upper,2
65,12954,2F10P,50.371400,-119.062100,NaN,50.371360,-119.062000,1839.0,,Silverstar Mountain,2
66,13009,1C29P,49.856000,-120.862900,NaN,49.855951,-120.862891,1460.0,,Shovelnose,2
67,13026,15394,49.166090,-120.567840,NaN,49.166090,-120.567840,988.0,,Similkameen Falls,2
68,13016,21192,51.669770,-119.565750,NaN,51.669770,-119.565750,545.0,,Mad River,2
69,13017,33095,49.148180,-118.530210,NaN,49.148180,-118.530210,998.0,,Eholt,2


In [8]:
check_sql = sa.text("""
SELECT
    station_id,
    station_name,
    lat,
    lon,
    elev
FROM meta_history
WHERE station_id = :station_id
""")


with engine.begin() as conn:
    for _, row in df_new.iterrows():

        station_id = int(row["Station ID"])

        old_lat  = row["old_lat"]
        old_lon  = row["old_lon"]
        old_elev = row["old_elev"]
        old_name = row["old_name"]

        new_lat  = row["new_lat"]
        new_lon  = row["new_lon"]
        new_elev = row["new_elev"]
        new_name = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 2. Compare with expected OLD values ---
        def norm(x):
            if pd.isna(x):
                return None
            return round(float(x), 6)

        def same(a, b):
            return norm(a) == norm(b)

        if not (
            same(db_row.lat,  old_lat) and
            same(db_row.lon,  old_lon) and
            same(db_row.elev, old_elev)
        ) and db_row.station_name != old_name:
            print(
                f"⚠️ Station {station_id} skipped (DB, XLS values differ)\n"
                f"   DB : name = {db_row.station_name}, lat={db_row.lat}, lon={db_row.lon}, elev={db_row.elev}\n"
                f"   XLS: name = {old_name}, lat={old_lat}, lon={old_lon}, elev={old_elev}"
            )

        else:
            print(
                f"✅ Station {station_id}, all match \n"
                f"   DB : name = {db_row.station_name}, lat={db_row.lat}, lon={db_row.lon}, elev={db_row.elev}"

            )


✅ Station 2299, all match 
   DB : name = HAGENSBORG II, lat=52.3872, lon=-126.5897, elev=50
✅ Station 1658, all match 
   DB : name = CHILLIWACK NURSERY, lat=49.0983, lon=-121.6633, elev=295
✅ Station 1675, all match 
   DB : name = MACHMELL (MB), lat=51.5933, lon=-126.4517, elev=366
✅ Station 1683, all match 
   DB : name = BOB QUINN LK, lat=56.9833, lon=-130.2533, elev=609
✅ Station 1710, all match 
   DB : name = INGENIKA PT, lat=56.9777, lon=-125.176, elev=1213
✅ Station 1722, all match 
   DB : name = VANDERHOOF, lat=54.0555, lon=-124.0102, elev=678
✅ Station 1724, all match 
   DB : name = GRASSY PLAINS, lat=53.9467, lon=-125.8667, elev=944
✅ Station 1746, all match 
   DB : name = ZZ PARROTT L/O-1250, lat=54.0183, lon=-126.385, elev=1259
✅ Station 1755, all match 
   DB : name = VALEMOUNT 1, lat=52.87, lon=-119.2973, elev=797
✅ Station 1789, all match 
   DB : name = TALCHAKO (MB), lat=52.2517, lon=-126.0283, elev=244
✅ Station 1794, all match 
   DB : name = CLEARWATER, lat=51

In [9]:
update_sql = sa.text("""
UPDATE meta_history
SET
    station_name  = :new_name,
    lat  = :new_lat,
    lon  = :new_lon,
    elev = :new_elev
WHERE station_id = :station_id
""")


with engine.begin() as conn:
    for _, row in df_new.iterrows():

        station_id = int(row["Station ID"])

        old_lat  = row["old_lat"]
        old_lon  = row["old_lon"]
        old_elev = row["old_elev"]
        old_name = row["old_name"]

        new_lat  = row["new_lat"]
        new_lon  = row["new_lon"]
        new_elev = row["new_elev"]
        new_name = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 3. Perform update ---
        result = conn.execute(
            update_sql,
            {
                "station_id": station_id,
                "new_name": new_name,
                "new_lat": new_lat,
                "new_lon": new_lon,
                "new_elev": new_elev,
            }
        )

        if result.rowcount == 1:
            print(
                f"✅ Updated station {station_id}: "
                f"({old_name}, {old_lat}, {old_lon}, {old_elev}) → "
                f"({new_name}, {new_lat}, {new_lon}, {new_elev})"
            )
        else:
            print(f"⚠️ Station {station_id}: no update applied")


✅ Updated station 2299: (HAGENSBORG II - HAGENSBORG 2, 52.3872, -126.5897, 50.0) → (HAGENSBORG II - HAGENSBORG 2, 52.3873, -126.5897, 47.0)
✅ Updated station 1658: (CHILLIWACK NURSERY, 49.0983, -121.6633, 295.0) → (CHILLIWACK NURSERY, 49.0953, -121.6628, 320.0)
✅ Updated station 1675: (MACHMELL (MB), 51.5933, -126.4517, 366.0) → (MACHMELL, 51.5931, -126.4508, 344.0)
✅ Updated station 1683: (BOB QUINN LK, 51.5933, -126.4517, 366.0) → (BOB QUINN LAKE, 51.5931, -126.4508, 344.0)
✅ Updated station 1710: (INGENIKA PT, 56.9777, -125.176, 1213.0) → (INGENIKA POINT, 57.0194, -125.1804, 680.0)
✅ Updated station 1722: (VANDERHOOF, 54.0555, -124.0102, 678.0) → (VANDERHOOF HUB, 54.0554, -124.0102, 681.0)
✅ Updated station 1724: (GRASSY PLAINS, 53.9467, -125.8667, 944.0) → (GRASSY PLAINS HUB, 53.9454, -125.8761, 938.0)
✅ Updated station 1746: (ZZ PARROTT L/O-1250, 54.0183, -126.385, 1259.0) → (PARROTT, 54.0186, -126.389, 1256.0)
✅ Updated station 1755: (VALEMOUNT 1, 52.87, -119.2973, 797.0) → (VALE

In [10]:
check_sql = sa.text("""
SELECT
    station_id,
    station_name,
    lat,
    lon,
    elev
FROM meta_history
WHERE station_id = :station_id
""")


with engine.begin() as conn:
    for _, row in df_new.iterrows():

        station_id = int(row["Station ID"])

        old_lat  = row["old_lat"]
        old_lon  = row["old_lon"]
        old_elev = row["old_elev"]
        old_name = row["old_name"]

        new_lat  = row["new_lat"]
        new_lon  = row["new_lon"]
        new_elev = row["new_elev"]
        new_name = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 2. Compare with expected OLD values ---
        def norm(x):
            if pd.isna(x):
                return None
            return round(float(x), 6)

        def same(a, b):
            return norm(a) == norm(b)

        if not (
            same(db_row.lat,  new_lat) and
            same(db_row.lon,  new_lon) and
            same(db_row.elev, new_elev)
        ) and db_row.station_name != new_name:
            print(
                f"⚠️ Station {station_id} skipped (DB, XLS values differ)\n"
                f"   DB : name = {db_row.station_name}, lat={db_row.lat}, lon={db_row.lon}, elev={db_row.elev}\n"
                f"   XLS: name = {new_name}, lat={new_lat}, lon={new_lon}, elev={new_elev}"
            )

        else:
            print(
                f"✅ Station {station_id}, all match \n"
                f"   DB : name = {db_row.station_name}, lat={db_row.lat}, lon={db_row.lon}, elev={db_row.elev}"

            )


✅ Station 2299, all match 
   DB : name = HAGENSBORG II - HAGENSBORG 2, lat=52.3873, lon=-126.5897, elev=47.0
✅ Station 1658, all match 
   DB : name = CHILLIWACK NURSERY, lat=49.0953, lon=-121.6628, elev=320.0
✅ Station 1675, all match 
   DB : name = MACHMELL, lat=51.5931, lon=-126.4508, elev=344.0
✅ Station 1683, all match 
   DB : name = BOB QUINN LAKE, lat=51.5931, lon=-126.4508, elev=344.0
✅ Station 1710, all match 
   DB : name = INGENIKA POINT, lat=57.0194, lon=-125.1804, elev=680.0
✅ Station 1722, all match 
   DB : name = VANDERHOOF HUB, lat=54.0554, lon=-124.0102, elev=681.0
✅ Station 1724, all match 
   DB : name = GRASSY PLAINS HUB, lat=53.9454, lon=-125.8761, elev=938.0
✅ Station 1746, all match 
   DB : name = PARROTT, lat=54.0186, lon=-126.389, elev=1256.0
✅ Station 1755, all match 
   DB : name = VALEMOUNT HUB, lat=52.8698, lon=-119.2991, elev=797.0
✅ Station 1789, all match 
   DB : name = TALCHAKO, lat=52.2514, lon=-126.0284, elev=306.0
✅ Station 1794, all match 
   